# OpenHands — architecture & live demo

Core in this repo: `openhands_all/app_server` (Python, **control plane only**). The real agent
loop runs in an **external, non-vendored `openhands-agent-server`** (`openhands.sdk`/`.tools`).
Condensed harness: `openhands_all/agent_harness/harness`. Write-up: `../01`, `../06`.

## Architecture in one screen

```

Client --HTTP--> app_server (this repo)

   1. provision sandbox running the agent-server

   2. assemble Agent + StartConversationRequest (LLM, tools, skills, hooks, secrets, policy)

   3. POST {agent_server}/api/conversations                (start turn)

        agent-server [EXTERNAL, in sandbox] runs the loop

   4. <--HTTP POST /api/v1/webhooks/events/{id}--           (events pushed back)

   5. persist events + run callbacks (title, analytics)

```

- **Orchestrator:** `live_status_app_conversation_service.py` — status machine +
  `_build_start_conversation_request_for_user` (`:1630`) + dispatch (`:486`).

- **Event sourcing:** append-only one-JSON-per-event (`event/event_service_base.py:save_event`).

- **Inbound webhook:** `event_callback/webhook_router.py:on_event` (`:468`).

- **Sandbox:** `sandbox/docker_sandbox_service.py` injects the callback URL `OH_WEBHOOKS_0_BASE_URL`.

- **Permission:** policy *selected* here (`_select_confirmation_policy:677`), *enforced* `[EXTERNAL]`.

- **Secrets:** `LookupSecret` + scoped JWT so tokens never enter the sandbox.

In [ ]:
import os, sys

def find_repo_root():
    markers = {'mini-agent', 'pi', 'openhands_all'}
    d = os.path.abspath(os.getcwd())
    while True:
        try:
            if markers.issubset(set(os.listdir(d))):
                return d
        except OSError:
            pass
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    for cand in ('/repo', os.path.expanduser('~')):
        if os.path.isdir(os.path.join(cand, 'mini-agent')):
            return cand
    raise RuntimeError('repo root not found; set REPO manually')

REPO = find_repo_root()
print('REPO =', REPO)

In [ ]:
# Helper: print a slice of a REAL core source file around a symbol, so every
# architectural claim in this notebook can be checked against the implementation.
def show(relpath, needle=None, ctx=6, max_lines=40):
    path = os.path.join(REPO, relpath)
    with open(path, encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
    print(f'# {relpath}  ({len(lines)} lines)')
    if needle is None:
        start, end = 0, min(max_lines, len(lines))
    else:
        idx = next((i for i, l in enumerate(lines) if needle in l), None)
        if idx is None:
            print(f'  (needle {needle!r} not found)')
            return
        start, end = max(0, idx - 1), min(len(lines), idx + ctx)
    for i in range(start, end):
        print(f'{i+1:5} | {lines[i].rstrip()}')

### Verify the control-plane / external split against the real code

There is **no model/tool loop** in this repo — only dispatch + webhook ingestion. Prove it:

In [ ]:
show('openhands_all/app_server/app_conversation/live_status_app_conversation_service.py',
     '/api/conversations', ctx=6)

In [ ]:
show('openhands_all/app_server/event_callback/webhook_router.py', 'def on_event', ctx=10)

In [ ]:
# The webhook callback URL injected into the sandbox (how events come back):
show('openhands_all/app_server/sandbox/sandbox_service.py', 'WEBHOOK_CALLBACK_VARIABLE', ctx=2)

## Live demo (condensed harness, offline)

The real loop is external, so we run the `agent_harness` reimplementation that reproduces the
**turn shape** OpenHands' agent-server would run: build context → one model call → detect tool
call → permission gate → execute → observation → re-iterate → final. Uses a rule-based
`EchoModelProvider` and `ConfirmRisky` (reads allowed, writes denied).

In [ ]:
sys.path.insert(0, os.path.join(REPO, 'openhands_all', 'agent_harness'))
from harness.impl import build_default_harness

h = build_default_harness(workspace=os.path.join(REPO, 'openhands_all', 'agent_harness'))
s = h.new_session()
reply = s.send('read README.md')   # EchoModelProvider turns this into a read_file tool call
print('FINAL:', reply.content[:120], '...\n')
print('EVENT TRACE (this is the event-sourced stream OpenHands would persist):')
for ev in h.trace.export(s.id):
    print('  ', ev.kind.value, ev.payload)

**Mapping to the real system:** each event above corresponds to an SDK `Event`
(`MessageEvent`/`ActionEvent`/`ObservationEvent`) that the external agent-server would POST to
`webhook_router.on_event`, which persists it via `event_service.save_event` and fans out
callbacks. The in-loop `PermissionPolicy.check` here is the agent-server's confirmation-policy
analogue — in real OpenHands the control plane only *selects and ships* that policy.